# VirtualiZarr → Icechunk in Cloud Storage → Append days
### Author: Eli Holmes modification of Rich Signell's notebook
[![Colab Badge](https://img.shields.io/badge/Open_in_Colab-blue?style=for-the-badge)][colab-link] [![Download Badge](https://img.shields.io/badge/Download-grey?style=for-the-badge)][download-link] [![JupyterHub](https://img.shields.io/badge/Jupyter_Hub-orange?style=for-the-badge)][jupyter-link]

[download-link]: https://github.com/nmfs-opensci/nmfshackdays-2026/blob/main/topics/2026-06-05/virtualizarr_ndvi_cdr_append-cloud.ipynb
[colab-link]: https://colab.research.google.com/github/nmfs-opensci/nmfshackdays-2026/blob/main/topics/2026-06-05/virtualizarr_ndvi_cdr_append-cloud.ipynb
[jupyter-link]: https://nmfs-openscapes.2i2c.cloud/hub/user-redirect/lab?fromURL=https://raw.githubusercontent.com/nmfs-opensci/nmfshackdays-2026/main/topics/2026-06-05/virtualizarr_ndvi_cdr_append-cloud.ipynb

##  Workflow

1. Open a single nc file on S3 and create a virtual dataset
2. Write virtual references to an Icechunk store in cloud storage
3. Open the next nc file on S3 and create a virtual dataset
4. Append to the Icechunk store
5. Repeat

In [ ]:
# Colab users, uncomment and run this
#!pip install -q icechunk  virtualizarr xarray obspec_utils obstore hvplot s3fs

In [2]:
%pip install -q icechunk virtualizarr obspec_utils 

  Using cached icechunk-2.0.6-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (12 kB)
Using cached icechunk-2.0.6-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (17.2 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [icechunk]2/3 [icechunk]
Note: you may need to restart the kernel to use updated packages.


In [3]:
import warnings
import shutil
from pathlib import Path

import xarray as xr
import icechunk
from obstore.store import from_url
from virtualizarr import open_virtual_dataset, open_virtual_mfdataset
from virtualizarr.parsers import HDFParser
from obspec_utils.registry import ObjectStoreRegistry

warnings.filterwarnings(
    "ignore",
    message="Numcodecs codecs are not in the Zarr version 3 specification*",
    category=UserWarning,
)

## Set up the urls to the REMOTE files in object storage

Start by looking at what is in the bucket and seeing how the data are organized.

In [4]:
import earthaccess
auth = earthaccess.login()
# are we authenticated?
if not auth.authenticated:
    # ask for credentials and persist them in a .netrc file
    auth.login(strategy="interactive", persist=True)

In [5]:
results = earthaccess.search_data(
    short_name = "PACE_OCI_L3M_CHL",
    temporal = ("2024-03-01", "2026-06-30"),
    granule_name="*.DAY.*.0p1deg.*"
)
len(results)

/srv/conda/envs/notebook/lib/python3.12/site-packages/earthaccess/results.py:348: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()


710

In [6]:
urls = [res.data_links(in_region=True)[0] for res in results]
urls[0:6]

['s3://ob-cumulus-prod-public/PACE_OCI.20240305.L3m.DAY.CHL.V3_1.chlor_a.0p1deg.nc',
 's3://ob-cumulus-prod-public/PACE_OCI.20240306.L3m.DAY.CHL.V3_1.chlor_a.0p1deg.nc',
 's3://ob-cumulus-prod-public/PACE_OCI.20240307.L3m.DAY.CHL.V3_1.chlor_a.0p1deg.nc',
 's3://ob-cumulus-prod-public/PACE_OCI.20240308.L3m.DAY.CHL.V3_1.chlor_a.0p1deg.nc',
 's3://ob-cumulus-prod-public/PACE_OCI.20240309.L3m.DAY.CHL.V3_1.chlor_a.0p1deg.nc',
 's3://ob-cumulus-prod-public/PACE_OCI.20240310.L3m.DAY.CHL.V3_1.chlor_a.0p1deg.nc']

## Setup S3 store, registry and Icechunk config for REMOTE files

Point `obstore` at the public NDVI bucket and register it so VirtualiZarr can resolve chunk references.

In [7]:
# Create an object-store handle for the REMOTE files.
from obstore.store import from_url
from obspec_utils.registry import ObjectStoreRegistry
from virtualizarr.parsers import HDFParser

bucket = "s3://ob-cumulus-prod-public/"

store = from_url(
    bucket,
    region="us-west-2",
    skip_signature=True,   # public / unsigned access
)

registry = ObjectStoreRegistry({bucket: store})
parser = HDFParser()

This will be passed to `icechunk` functions.

In [8]:
import icechunk

# Tell Icechunk how to access the remote virtual chunks
config = icechunk.RepositoryConfig.default()

config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        url_prefix="s3://ob-cumulus-prod-public/",
        store=icechunk.s3_store(
            region="us-west-2",
            anonymous=True,
        ),
    )
)

In [9]:
vds = open_virtual_dataset(
    url=urls[0],
    parser=parser,
    registry=registry,
    decode_times=True,
)

PermissionDeniedError: The operation lacked the necessary privileges to complete for path PACE_OCI.20240305.L3m.DAY.CHL.V3_1.chlor_a.0p1deg.nc: Error performing HEAD https://s3.us-west-2.amazonaws.com/ob-cumulus-prod-public/PACE_OCI.20240305.L3m.DAY.CHL.V3_1.chlor_a.0p1deg.nc in 24.15458ms - Server returned non-2xx status code: 403 Forbidden: 

Debug source:
PermissionDenied {
    path: "PACE_OCI.20240305.L3m.DAY.CHL.V3_1.chlor_a.0p1deg.nc",
    source: RetryError(
        RetryErrorImpl {
            method: HEAD,
            uri: Some(
                https://s3.us-west-2.amazonaws.com/ob-cumulus-prod-public/PACE_OCI.20240305.L3m.DAY.CHL.V3_1.chlor_a.0p1deg.nc,
            ),
            retries: 0,
            max_retries: 10,
            elapsed: 24.15458ms,
            retry_timeout: 180s,
            inner: Status {
                status: 403,
                body: Some(
                    "",
                ),
            },
        },
    ),
}

### Set up the Icechunk storage bucket

Source Coop makes it easy to get this info via a 'View Credentials' link. Since I am working in a Jupyter notebook, I will load the variables via a json file. 

1. Create a json file and add to `.gitignore` so you do not accidentally commit it to GitHub. Never hard-code tokens into notebooks.
2. Get the bucket info: provider (S3, GCP, etc), bucket, prefix and pass that to set up the storage object.

![](sc_env_var.png)

In [5]:
# Read in the json file
import json
from pathlib import Path

with open("source-creds.json") as f:
    source_creds = json.load(f)

In [6]:
# This info you get from your storage
source_bucket = "us-west-2.opendata.source.coop"
source_prefix = "eeholmes/chlaz/icechunk"
source_region = "us-west-2"

storage = icechunk.s3_storage(
    bucket=source_bucket,
    prefix=source_prefix,
    region=source_creds["region_name"],
    access_key_id=source_creds["aws_access_key_id"],
    secret_access_key=source_creds["aws_secret_access_key"],
    session_token=source_creds["aws_session_token"],
)

In [7]:
# Create store if it is empty
try:
    repo = icechunk.Repository.create(storage, config)
    print("Created new Icechunk repo")
except Exception:
    repo = icechunk.Repository.open(storage, config=config)
    print("Opened existing Icechunk repo")

# Create a session
session = repo.writable_session(branch="main")

Created new Icechunk repo


## Once set up, the rest is the same

In [8]:
# Run the for loop
import time
from pathlib import Path

for i, url in enumerate(urls[:5]):
    filename = Path(url).name
    start = time.perf_counter()

    print(f"Adding {filename}")

    vds = open_virtual_dataset(
        url=url,
        parser=parser,
        registry=registry,
        loadable_variables=["time", "latitude", "longitude"],
        decode_times=True,
    ).drop_vars(["nv", "ncrs", "crs"], errors="ignore")

    if i == 0:
        vds.vz.to_icechunk(session.store)
    else:
        vds.vz.to_icechunk(session.store, append_dim="time")

    elapsed = time.perf_counter() - start
    print(f"Finished {filename} in {elapsed:.2f} seconds")

snapshot_id = session.commit("5 days NDVI CDR Jan 2000")
print("Committed:", snapshot_id)

Adding AVHRR-Land_v005_AVH13C1_NOAA-14_20000101_c20170623095628.nc
Finished AVHRR-Land_v005_AVH13C1_NOAA-14_20000101_c20170623095628.nc in 8.03 seconds
Adding AVHRR-Land_v005_AVH13C1_NOAA-14_20000102_c20170623101557.nc
Finished AVHRR-Land_v005_AVH13C1_NOAA-14_20000102_c20170623101557.nc in 6.04 seconds
Adding AVHRR-Land_v005_AVH13C1_NOAA-14_20000103_c20170623103338.nc
Finished AVHRR-Land_v005_AVH13C1_NOAA-14_20000103_c20170623103338.nc in 5.64 seconds
Adding AVHRR-Land_v005_AVH13C1_NOAA-14_20000104_c20170623105028.nc
Finished AVHRR-Land_v005_AVH13C1_NOAA-14_20000104_c20170623105028.nc in 6.30 seconds
Adding AVHRR-Land_v005_AVH13C1_NOAA-14_20000105_c20170623110559.nc
Finished AVHRR-Land_v005_AVH13C1_NOAA-14_20000105_c20170623110559.nc in 5.88 seconds
Committed: A59D4CBRS3XT6MHPGZ50


In [25]:
# So about 24 hours to do all 40 years
# If done in series, not parallel
365*40*6/(60*60)

24.333333333333332

## Public reading of our icechunk

Add these as instructions in for your dataset.

In [1]:
import icechunk
import xarray as xr

# -------------------------------------------------------------------
# 1. Authorize access to the original NOAA NetCDF chunks
# -------------------------------------------------------------------
# The Icechunk repo contains virtual chunk references back to a
# public NOAA S3 bucket. 
credentials = icechunk.containers_credentials({
    "s3://noaa-cdr-ndvi-pds/": icechunk.s3_credentials(anonymous=True)
})

# -------------------------------------------------------------------
# 2. Tell Icechunk where virtual chunks are allowed to come from
# -------------------------------------------------------------------
config = icechunk.RepositoryConfig.default()
config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        url_prefix="s3://noaa-cdr-ndvi-pds/",
        store=icechunk.s3_store(region="us-east-1", anonymous=True),
    ),
)

# -------------------------------------------------------------------
# 3. Point to the public Icechunk repo on Source Cooperative
# -------------------------------------------------------------------
# This is the location of the Icechunk repository itself.
storage = icechunk.s3_storage(
    bucket="us-west-2.opendata.source.coop",
    prefix="eeholmes/chlaz/icechunk",
    region="us-west-2",
    anonymous=True,
)

# -------------------------------------------------------------------
# 4. Open the Icechunk repo and read it with xarray
# -------------------------------------------------------------------
repo = icechunk.Repository.open(
    storage,
    config=config,
    authorize_virtual_chunk_access=credentials,
)

session = repo.readonly_session(branch="main")

ds = xr.open_zarr(
    session.store,
    consolidated=False,
    chunks=None,
)

ds

/srv/conda/envs/notebook/lib/python3.12/site-packages/zarr/codecs/numcodecs/_codecs.py:141: ZarrUserWarning: Numcodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  super().__init__(**codec_config)


<xarray.Dataset> Size: 2GB
Dimensions:    (time: 5, latitude: 3600, longitude: 7200, nv: 2)
Coordinates:
  * time       (time) datetime64[ns] 40B 2000-01-01 2000-01-02 ... 2000-01-05
  * latitude   (latitude) float32 14kB 89.97 89.93 89.88 ... -89.93 -89.97
  * longitude  (longitude) float32 29kB -180.0 -179.9 -179.9 ... 179.9 180.0
Dimensions without coordinates: nv
Data variables:
    NDVI       (time, latitude, longitude) float64 1GB ...
    QA         (time, latitude, longitude) int16 259MB ...
    lat_bnds   (latitude, nv) float32 29kB ...
    TIMEOFDAY  (time, latitude, longitude) datetime64[ns] 1GB ...
    lon_bnds   (longitude, nv) float32 58kB ...
Attributes: (12/48)
    title:                                  Normalized Difference Vegetation ...
    institution:                            NASA/GSFC/SED/ESD/HBSL/TIS/MODIS-...
    Conventions:                            CF-1.6, ACDD-1.3
    standard_name_vocabulary:               CF Standard Name Table (v25, 05 J...
    naming_authority:                       gov.noaa.ncei
    license:                                See the Use Agreement for this CD...
    ...                                     ...
    PercentValidDaytimeData:                32.01
    PercentValidDaytimeLand:                32.01
    PercentValidClearDaytimeLand:           3.14
    PercentValidDaytimeLandInCloudShadow:   1.04
    PercentValidClearDaytimeWater:          0.00
    PercentValidDaytimeWaterInCloudShadow:  0.00

## We can vizualize Icechunks in public stores

Javascript libraries `zarrita` and `icechunk-js` allow us to create visualizations of public Zarr stores.

https://eeholmes.github.io/gridlook/#https://data.source.coop/eeholmes/chlaz/icechunk

![](ndvi_gridlook.png)

## Make a plot with hvplot

In [10]:
import hvplot.xarray

ds["NDVI"].isel(time=0).hvplot.quadmesh(
    rasterize=True,
    x="longitude",
    y="latitude",
    title="AVHRR NDVI — 2000-01-01",
    width=800,
    height=400,
    xlim=(-180, 180),
    ylim=(-90, 90),
    cmap = "turbo_r"
)

:DynamicMap   []
   :Image   [longitude,latitude]   (NOAA Climate Data Record of Normalized Difference Vegetation Index)

## A non-memory efficient plot

In [ ]:
# This would require a lot of RAM
ds["NDVI"].isel(time=0).plot(
    x="longitude",
    y="latitude",
    figsize=(10, 4)
)